In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

edgar_dir = "/content/drive/MyDrive/EDGAR/extracted_edgar"
earnings_dir = "/content/drive/MyDrive/EarningsCall/extracted_earnings"

edgar_subfolders = []
earnings_subfolders = []

for entry in os.listdir(edgar_dir):
    path = os.path.join(edgar_dir, entry)
    if os.path.isdir(path):
        edgar_subfolders.append(entry)

for entry in os.listdir(earnings_dir):
    path = os.path.join(earnings_dir, entry)
    if os.path.isdir(path):
        earnings_subfolders.append(entry)


print(edgar_subfolders)
print(earnings_subfolders)

['AMD', 'AAPL', 'AMAT', 'ANET', 'CSCO', 'GLW', 'KLAC', 'LRCX', 'MCHP', 'MU', 'MPWR', 'NVDA', 'ON', 'QCOM', 'SMCI', 'TDY', 'TER', 'AMGN', 'TECH', 'DHR', 'ISRG', 'ABT', 'LLY', 'MDT', 'MRK', 'MTD', 'PFE', 'VTRS', 'WAT', 'ZTS', 'APTV', 'F', 'GM', 'TSLA', 'NOVN', 'RACE', 'STLAM', 'ASML', 'SAP', 'ERICB', 'LOGN', 'NOKIA', 'STMMI']
['.ipynb_checkpoints', 'ABT', 'AMGN', 'AMD', 'MBG', 'AMAT', 'ANET', 'ASML', 'TECH', 'APTV', 'AZN', 'BMW', 'DHR', 'CSCO', 'GLW', 'LLY', 'F', 'GM', 'RACE', 'MRK']


In [ ]:
import os

edgar_files_dict = {}

for subfolder in edgar_subfolders:
    subfolder_path = os.path.join(edgar_dir, subfolder)
    files_in_subfolder = []
    if os.path.isdir(subfolder_path):
        for item in os.listdir(subfolder_path):
            item_path = os.path.join(subfolder_path, item)
            if os.path.isfile(item_path):
                files_in_subfolder.append(item)
    edgar_files_dict[subfolder] = files_in_subfolder

# Display the first few entries or a summary
print(f"Number of subfolders processed: {len(edgar_files_dict)}")
for key, value in list(edgar_files_dict.items())[:5]: # Display first 5 entries
    print(f"\nSubfolder: {key}")
    print(f"  Files: {value[:5]}...") # Display first 5 files if many

Number of subfolders processed: 43

Subfolder: AMD
  Files: ['mda_2025_10-K_0000002488-25-000012.txt.txt', 'mda_2024_10-K_0000002488-24-000012.txt.txt', 'mda_2020_10-K_0000002488-20-000008.txt.txt', 'mda_2022_10-K_0000002488-22-000016.txt.txt', 'mda_2021_10-K_0001628280-21-001185.txt.txt']...

Subfolder: AAPL
  Files: ['mda_2025_10-K_0000320193-25-000079.txt.txt', 'mda_2021_10-K_0000320193-21-000105.txt.txt', 'mda_2023_10-K_0000320193-23-000106.txt.txt', 'mda_2022_10-K_0000320193-22-000108.txt.txt', 'mda_2024_10-K_0000320193-24-000123.txt.txt']...

Subfolder: AMAT
  Files: ['mda_2022_10-K_0000006951-22-000043.txt.txt', 'mda_2025_10-K_0001628280-25-056742.txt.txt', 'mda_2024_10-K_0000006951-24-000044.txt.txt', 'mda_2023_10-K_0000006951-23-000041.txt.txt', 'mda_2020_10-K_0000006951-20-000048.txt.txt']...

Subfolder: ANET
  Files: ['mda_2024_10-K_0001596532-24-000043.txt.txt', 'mda_2025_10-K_0001596532-25-000028.txt.txt', 'mda_2023_10-K_0001596532-23-000016.txt.txt', 'mda_2018_10-K_000159

In [ ]:
import pandas as pd
import nltk

# Ensure nltk punkt tokenizer is downloaded
nltk.download('punkt')
nltk.download('punkt_tab')

keywords = ["China", "Chinese", "PRC"]
chinese_cities = [
    "Beijing", "Shanghai", "Chongqing", "Tianjin", "Guangzhou",
    "Shenzhen", "Chengdu", "Wuhan", "Hangzhou", "Nanjing",
    "Xi'an", "Suzhou", "Zhengzhou", "Changsha", "Shenyang",
    "Qingdao", "Ningbo", "Kunming", "Wuxi", "Hefei",
    "Fuzhou", "Xiamen", "Harbin", "Jinan", "Dalian"
]

# Combine into a single set of terms to check for faster lookup or iteration
search_terms = set(keywords + chinese_cities)

extracted_data = []

for company, files in edgar_files_dict.items():
    company_path = os.path.join(edgar_dir, company)

    for filename in files:
        file_path = os.path.join(company_path, filename)

        try:
            # Attempt to extract metadata from filename
            # Example format: mda_2025_10-K_0000002488-25-000012.txt.txt
            parts = filename.split('_')
            if len(parts) >= 3:
                year = parts[1]
                doc_type = parts[2]
            else:
                year = "Unknown"
                doc_type = "Unknown"

            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()

            # Tokenize into sentences
            sentences = nltk.sent_tokenize(text)

            for sentence in sentences:
                # Check if any search term is in the sentence
                # Using explicit iteration for clarity.
                # Note: This is case-sensitive based on user list.
                # If you need case-insensitive, we can convert both to lower.
                found = False
                for term in search_terms:
                    if term in sentence:
                        found = True
                        break

                if found:
                    extracted_data.append({
                        'company': company,
                        'year': year,
                        'document_type': doc_type,
                        'sentence': sentence,
                        'filename': filename
                    })

        except Exception as e:
            print(f"Error processing file {filename}: {e}")

# Create DataFrame
df_sentences = pd.DataFrame(extracted_data)

# Display the first few rows
print(f"Extracted {len(df_sentences)} sentences.")
display(df_sentences.head())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Extracted 2691 sentences.


,company,year,document_type,sentence,filename
0,AMD,2020,10-K,We conduct product and system research and dev...,mda_2020_10-K_0000002488-20-000008.txt.txt
1,AMD,2020,10-K,"For example, the European Union (EU) and China...",mda_2020_10-K_0000002488-20-000008.txt.txt
2,AMD,2020,10-K,"A number of jurisdictions including the EU, Au...",mda_2020_10-K_0000002488-20-000008.txt.txt
3,AMD,2022,10-K,We conduct product and system research and dev...,mda_2022_10-K_0000002488-22-000016.txt.txt
4,AMD,2022,10-K,"For example, the European Union (EU) and China...",mda_2022_10-K_0000002488-22-000016.txt.txt


In [ ]:
df_sentences

,company,year,document_type,sentence,filename
0,AMD,2020,10-K,We conduct product and system research and dev...,mda_2020_10-K_0000002488-20-000008.txt.txt
1,AMD,2020,10-K,"For example, the European Union (EU) and China...",mda_2020_10-K_0000002488-20-000008.txt.txt
2,AMD,2020,10-K,"A number of jurisdictions including the EU, Au...",mda_2020_10-K_0000002488-20-000008.txt.txt
3,AMD,2022,10-K,We conduct product and system research and dev...,mda_2022_10-K_0000002488-22-000016.txt.txt
4,AMD,2022,10-K,"For example, the European Union (EU) and China...",mda_2022_10-K_0000002488-22-000016.txt.txt
...,...,...,...,...,...
2686,STMMI,2014,20-F,Mr. Gualandris has authored several technical ...,mda_2014_20-F_0001193125-14-084249.txt.txt
2687,STMMI,2014,20-F,François Guibert is Executive Vice President o...,mda_2014_20-F_0001193125-14-084249.txt.txt
2688,STMMI,2014,20-F,Guibert has led ST’s operations in Asia Pacifi...,mda_2014_20-F_0001193125-14-084249.txt.txt
2689,STMMI,2014,20-F,Guibert serves as Director of ST’s JV with She...,mda_2014_20-F_0001193125-14-084249.txt.txt


In [ ]:
output_filename = 'extracted_sentences.xlsx'
df_sentences.to_excel(output_filename, index=False)
print(f"DataFrame saved to {output_filename}")

DataFrame saved to extracted_sentences.xlsx
